# Ejercicio 3 — Residual U-Net ligera

Red de segmentación binaria inspirada en U-Net y bloques residuales. Se comprueba la arquitectura sin entrenarla ni descargar datos o pesos.

Ejecuta todas las celdas con un kernel Python que tenga PyTorch instalado, iniciando Jupyter desde la raíz del proyecto o desde `03_unet/` (entrega) o `ejercicios/03_unet/` (proyecto). Mantén `model.py` junto a este notebook.

In [1]:
import importlib.util
from pathlib import Path
import torch

# Ruta relativa al directorio de trabajo, sin importar el módulo genérico `model`.
folder = Path.cwd()
if not (folder / "model.py").is_file():
    folder = folder / "03_unet" if (folder / "03_unet" / "model.py").is_file() else folder / "ejercicios" / "03_unet"
model_path = folder / "model.py"
if not model_path.is_file():
    raise FileNotFoundError("Inicia Jupyter desde la raíz del proyecto o ejercicios/03_unet/.")
spec = importlib.util.spec_from_file_location("residual_unet_ejercicio_03", model_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
ResidualBlock, ResidualUNet = module.ResidualBlock, module.ResidualUNet

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResidualUNet().to(device).eval()
print(f"Dispositivo: {device}")

Dispositivo: cuda


## Diseño

El encoder extrae características a distintas escalas; el bottleneck amplía el contexto. El decoder interpola y concatena las características del encoder, recuperando información espacial. Cada bloque especial contiene dos convoluciones 3×3, normalización, activación y una suma residual; usa una proyección 1×1 cuando cambian los canales. La ruta residual facilita el flujo de gradientes y el aprendizaje de transformaciones residuales. La interpolación al tamaño del skip soporta dimensiones impares. Estas son decisiones de implementación propias, no una reproducción exacta de las referencias.

La capa final 1×1 devuelve **logits**, sin sigmoid: para probabilidades se puede aplicar `torch.sigmoid` después. La red sin entrenar no produce una segmentación significativa.

Las skip connections de U-Net transportan información espacial del encoder al decoder; las conexiones residuales facilitan el flujo de gradientes dentro de cada bloque. Son mecanismos distintos y complementarios. La red es ligera, pero su precisión de segmentación no puede evaluarse sin entrenamiento y datos etiquetados.

In [2]:
# Verifica la proyección 1×1 y la ruta identidad del bloque especial.
assert isinstance(ResidualBlock(3, 16).projection, torch.nn.Conv2d)
assert isinstance(ResidualBlock(16, 16).projection, torch.nn.Identity)
params = sum(p.numel() for p in model.parameters())
assert 0 < params < 1_000_000, f"Número inesperado de parámetros: {params}"
print(f"Parámetros: {params:,} (< 1 millón; arquitectura ligera)")

with torch.inference_mode():
    for h, w in [(128, 128), (129, 131)]:
        x = torch.randn(1, 3, h, w, device=device)
        y = model(x)
        y_repeat = model(x)
        assert y.shape == (1, 1, h, w), (y.shape, h, w)
        assert torch.isfinite(y).all(), "Se encontraron valores no finitos"
        assert torch.allclose(y, y_repeat, rtol=1e-5, atol=1e-6), "Inferencia inconsistente"
        print(f"{tuple(x.shape)} -> {tuple(y.shape)} | finitos: sí | repetición consistente: sí")

Parámetros: 125,041 (< 1 millón; arquitectura ligera)


(1, 3, 128, 128) -> (1, 1, 128, 128) | finitos: sí | repetición consistente: sí
(1, 3, 129, 131) -> (1, 1, 129, 131) | finitos: sí | repetición consistente: sí


## Comprobación de gradientes

Pérdida sintética y retropropagación sin optimizador ni actualización de pesos: esta prueba no entrena ni evalúa la precisión de segmentación.

Se comprueban únicamente los gradientes de la primera convolución y la proyección de `enc1`, no los de todos los parámetros.

In [3]:
model.zero_grad(set_to_none=True)
x_grad = torch.randn(1, 3, 32, 32, device=device)
loss = model(x_grad).square().mean()
loss.backward()
for name, parameter in [("rama convolucional", model.enc1.conv1.weight),
                        ("proyección 1×1", model.enc1.projection.weight)]:
    assert parameter.grad is not None, f"Falta gradiente: {name}"
    assert torch.isfinite(parameter.grad).all(), f"Gradiente no finito: {name}"
    print(f"{name}: gradiente presente y finito")
model.zero_grad(set_to_none=True)

rama convolucional: gradiente presente y finito
proyección 1×1: gradiente presente y finito


## Referencias

- Ronneberger, O., Fischer, P. y Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI. https://arxiv.org/abs/1505.04597
- He, K., Zhang, X., Ren, S. y Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR. https://arxiv.org/abs/1512.03385
- Zhang, Z., Liu, Q. y Wang, Y. (2018). *Road Extraction by Deep Residual U-Net*. IEEE Geoscience and Remote Sensing Letters. https://arxiv.org/abs/1711.10684